# BrainTumorAI Google Colab Backend
This notebook runs the BrainTumorAI FastAPI server directly on Google Colab's GPU.
**Instructions:**
1. Ensure you are using a **T4 GPU** runtime (Runtime -> Change runtime type -> Hardware accelerator: T4 GPU).
2. Press **Run all** (Cmd/Ctrl + F9). The notebook will automatically clone the codebase, setup models, and start the API.

In [ ]:
import os
import subprocess
import sys

print("=========================================")
print(" 1. REPOSITORY SETUP")
print("=========================================")

# Verify git and git-lfs
try:
    subprocess.run(["git", "--version"], check=True, capture_output=True)
except FileNotFoundError:
    sys.exit("CRITICAL ERROR: git is not installed.")

try:
    subprocess.run(["git", "lfs", "version"], check=True, capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("git-lfs not found. Installing git-lfs...")
    subprocess.run(["apt-get", "install", "-y", "git-lfs"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["git", "lfs", "install"], check=True, stdout=subprocess.DEVNULL)

repo_url = "https://github.com/BiswasApurbo/BrainTumorAI.git"
repo_dir = "/content/BrainTumorAI"

if not os.path.exists(repo_dir):
    print("Cloning repository (with LFS files)...")
    result = subprocess.run(["git", "clone", repo_url, repo_dir], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        sys.exit("CRITICAL ERROR: Failed to clone repository.")
else:
    print("Repository already exists. Pulling latest changes...")
    result = subprocess.run(["git", "pull"], cwd=repo_dir, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        sys.exit("CRITICAL ERROR: Failed to pull latest changes.")

# Change directory
os.chdir(repo_dir)

# Print latest commit
commit_hash = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"Repository ready. Latest commit: {commit_hash}")


In [ ]:
import os
import sys
import subprocess

print("\n=========================================")
print(" 2. DEPENDENCY & GPU SETUP")
print("=========================================")

print(f"Python Version: {sys.version.split()[0]}")

print("Installing 'uv' package manager...")
result = subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr)
    sys.exit("CRITICAL ERROR: Failed to install uv.")
    
os.environ["PATH"] += ":/root/.cargo/bin"

print("Syncing project dependencies (this may take a minute). Logs printed on failure...")
result = subprocess.run(["uv", "sync"], capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    sys.exit("CRITICAL ERROR: Dependency installation failed.")
print("Dependencies installed successfully.")

verify_script = """
import torch
import sys

print(f'PyTorch Version: {torch.__version__}')
cuda_available = torch.cuda.is_available()

if not cuda_available:
    sys.exit("CRITICAL ERROR: CUDA is not available. Ensure you are using a T4 GPU runtime.")

print(f'CUDA Version: {torch.version.cuda}')
print(f'GPU Model: {torch.cuda.get_device_name(0)}')

# Memory
total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'Available GPU Memory: {total_memory:.1f} GB')
"""
print("\nVerifying GPU capabilities...")
result = subprocess.run(["uv", "run", "python", "-c", verify_script], capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr)
    sys.exit(result.stderr if result.stderr else "CRITICAL ERROR: CUDA Verification failed.")
print(result.stdout.strip())


In [ ]:
import os
import sys

print("\n=========================================")
print(" 3. MODEL VERIFICATION & ENVIRONMENT")
print("=========================================")

local_models_path = '/content/BrainTumorAI/models'

# 1. Verify nnUNet
nnunet_checkpoint = os.path.join(
    local_models_path,
    'nnUNet/3d_fullres/Task082_BraTS2020/nnUNetTrainerV2BraTSRegions_DA4_BN__nnUNetPlansv2.1_bs5/fold_0/model_final_checkpoint.model'
)

if not os.path.exists(nnunet_checkpoint):
    sys.exit(f"CRITICAL ERROR: nnUNet checkpoint missing at {nnunet_checkpoint}")
    
nnunet_size = os.path.getsize(nnunet_checkpoint)
if nnunet_size < 100 * 1024 * 1024:  # Less than 100MB indicates an LFS stub
    sys.exit(f"CRITICAL ERROR: nnUNet checkpoint is only {nnunet_size} bytes (Git LFS pointer stub). Real weights required.")
    
if not os.access(nnunet_checkpoint, os.R_OK):
    sys.exit(f"CRITICAL ERROR: nnUNet checkpoint at {nnunet_checkpoint} is not readable.")

# 2. Verify SynthSeg
synthseg_checkpoint = os.path.join(
    local_models_path,
    'SynthSeg/models/synthseg_1.0.h5'
)

if not os.path.exists(synthseg_checkpoint):
    sys.exit(f"CRITICAL ERROR: SynthSeg checkpoint missing at {synthseg_checkpoint}")
    
synthseg_size = os.path.getsize(synthseg_checkpoint)
if synthseg_size < 10 * 1024 * 1024: # Less than 10MB indicates an LFS stub
    sys.exit(f"CRITICAL ERROR: SynthSeg checkpoint is only {synthseg_size} bytes (Git LFS pointer stub). Real weights required.")

if not os.access(synthseg_checkpoint, os.R_OK):
    sys.exit(f"CRITICAL ERROR: SynthSeg checkpoint at {synthseg_checkpoint} is not readable.")

print(f"✓ nnUNet Checkpoint: {nnunet_checkpoint}")
print(f"  Size: {nnunet_size / (1024*1024):.1f} MB | Readable: True")
print(f"✓ SynthSeg Checkpoint: {synthseg_checkpoint}")
print(f"  Size: {synthseg_size / (1024*1024):.1f} MB | Readable: True")

# 3. Environment Variables
os.environ['RESULTS_FOLDER'] = local_models_path
os.environ['nnUNet_raw_data_base'] = os.path.join(local_models_path, 'nnUNet_raw_data_base')
os.environ['nnUNet_preprocessed'] = os.path.join(local_models_path, 'nnUNet_preprocessed')

print("\nConfigured Environment Variables:")
print(f"RESULTS_FOLDER={os.environ['RESULTS_FOLDER']}")
print(f"nnUNet_raw_data_base={os.environ['nnUNet_raw_data_base']}")
print(f"nnUNet_preprocessed={os.environ['nnUNet_preprocessed']}")


In [ ]:
import subprocess
import time
import requests
import os
import sys

print("\n=========================================")
print(" 4. START FastAPI BACKEND")
print("=========================================")

env = dict(os.environ)
env["PYTHONPATH"] = "src"

log_file_path = "server.log"
log_file = open(log_file_path, "w")

print("Starting uvicorn server in the background...")
server_process = subprocess.Popen(
    ["uv", "run", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Polling http://127.0.0.1:8000/health (timeout 120s)...")
healthy = False
start_time = time.time()
while time.time() - start_time < 120:
    # If the process crashed, no point in waiting
    if server_process.poll() is not None:
        break

    try:
        response = requests.get("http://127.0.0.1:8000/health")
        if response.status_code == 200:
            healthy = True
            break
    except requests.ConnectionError:
        pass
    time.sleep(2)

if not healthy:
    server_process.terminate()
    log_file.close()
    print("\nCRITICAL ERROR: FastAPI backend failed to start or did not become healthy.")
    print("----- LAST 100 LINES OF SERVER LOG -----")
    with open(log_file_path, "r") as f:
        lines = f.readlines()
        for line in lines[-100:]:
            print(line, end="")
    print("----------------------------------------")
    sys.exit("Server startup failed. Check the logs above.")

print("FastAPI is healthy and running!")


In [ ]:
print("\n=========================================")
print(" 5. LAUNCH WEB TUNNEL & VERIFY")
print("=========================================")

print("Installing pyngrok...")
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok", "-q"], check=True)

from pyngrok import ngrok, exception
import time
import requests
import torch

try:
    public_url = ngrok.connect(8000).public_url
except exception.PyngrokNgrokError as e:
    print(f"\nCRITICAL ERROR: ngrok authentication or connection failed.")
    print(f"Details: {e}")
    sys.exit("Failed to create ngrok tunnel.")

local_url = "http://127.0.0.1:8000"

print("Verifying API externally (lightweight smoke test)...")
try:
    res = requests.get(f"{local_url}/health")
    if res.status_code == 200:
        smoke_test = "PASS (HTTP 200)"
    else:
        smoke_test = f"FAIL (HTTP {res.status_code})"
except Exception as e:
    smoke_test = f"FAIL ({e})"

print("\n=========================================")
print("             STARTUP SUMMARY             ")
print("=========================================")
print("✓ Repository: /content/BrainTumorAI")
print(f"✓ Python:     {sys.version.split()[0]}")
print(f"✓ Torch:      {torch.__version__}")
print(f"✓ CUDA:       {torch.version.cuda}")
print(f"✓ GPU:        {torch.cuda.get_device_name(0)}")
print("✓ Models:     Verified nnUNet & SynthSeg Checkpoints")
print("✓ Env Vars:   Configured successfully")
print("✓ FastAPI:    Running locally on port 8000")
print(f"✓ API Smoke:  {smoke_test}")
print("✓ ngrok:      Tunnel established successfully")
print("=========================================\n")

print(f"Local URL:  {local_url}")
print(f"Public URL: {public_url}\n")
print(f">>> OPEN THIS URL IN YOUR BROWSER: {public_url} <<<\n")

# Keep the cell running to prevent the tunnel and server from dying
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.kill()
